In [ ]:
import numpy as np
import pandas as pd
import json
from datetime import timedelta
import pycountry
from numpyro import distributions as dist
import arviz as az
from matplotlib import pyplot as plt
from matplotlib.pyplot import subplots

from emu_renewal.constants import DATA_PATH, DATE_FORMAT, RUN_DATA_DELAY, RTINIT_SD
from emu_renewal.inputs import get_cont_of_country, get_country_pop
from emu_renewal.renew import MultiStrainModel
from emu_renewal.run import find_run_start_time, find_run_end_time, get_country_vars, get_alpha_info, get_delta_info, get_mobility_provider, run_calibration
from emu_renewal.targets import SharedDispTarget

In [ ]:
countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
country = pycountry.countries.lookup("France")
mob_source = "fb_singletile_mob"
iso3 = country.alpha_3
continent = get_cont_of_country(iso3)
pop = get_country_pop(iso3)
data_start = find_run_start_time(pop, iso3)
end_time = find_run_end_time(iso3, mob_source)
run_start = data_start - timedelta(RUN_DATA_DELAY)
start_str = run_start.strftime(DATE_FORMAT)
end_str = data_start.strftime(DATE_FORMAT)
var_data = get_country_vars(iso3)
delta_var, delta_targ, delta_seed = get_delta_info(iso3, var_data, continent, end_time)
alpha_var, alpha_targ, alpha_seed = get_alpha_info(iso3, var_data, continent, end_time, delta_targ)
start_var = "eu"
var_names = [start_var] + alpha_var
seed_times = [] + alpha_seed
mob_provider = get_mobility_provider(iso3, mob_source)
model = MultiStrainModel(pop, run_start, end_time, var_names, seed_times, mob_provider, False)

In [ ]:
# proc = np.zeros(62)
proc = np.random.normal(0.0, 0.05, 62)
params = {
    "proc": proc,
    "gen_mean": 4.0,
    "gen_sd": 2.0,
    "cdr": 0.4,
    "ifr": 0.008,
    "rt_init": 0.9,
    "report_mean": 8.0,
    "report_sd": 3.0,
    "death_mean": 25.0,
    "death_sd": 3.0,
    "admit_mean": 15.0,
    "admit_sd": 2.0,
    "stay_mean": 7.0,
    "stay_sd": 1.0,
    "har": 0.035,
    "icu_admit_mean": 10.0,
    "icu_admit_sd": 1.0,
    "icu_stay_mean": 7.0,
    "icu_stay_sd": 1.0,
    "icuar": 0.02,
    "cross_immunity": 0.5,
    "seed_rates": np.array([-13.0, -13.0]),
    "relinfect": np.array([1.2]),
    "seed_offsets": np.array([50.0]),
    "vacc_protect_hosp": 0.0,
    "vacc_protect_death": 0.0,
    "mob_exp": 1.0,
    "shared_dispersion": 0.3,
}
results = model.renewal_func(**params)

In [ ]:
times = model.epoch.number_to_datetime(pd.Series(model.model_times))
death_vals = pd.Series(results["weekly_deaths"], index=times)[::7]
death_targ = SharedDispTarget(death_vals, weight=20)

In [ ]:
rt_prior = {"rt_init": dist.TruncatedNormal(0.0, RTINIT_SD, low=0.2)}
mob_exp_prior = {"mob_exp": dist.TruncatedNormal(0.0, 1.0, low=0.2)}

In [ ]:
death_vals.plot()

In [ ]:
calib, mcmc = run_calibration(model, params | rt_prior | mob_exp_prior, {"weekly_deaths": death_targ}, True)

In [ ]:
idata = az.from_numpyro(mcmc)

In [ ]:
def get_process_from_updates(rt_init, updates):
    return pd.Series(np.cumsum(np.concatenate([np.array([rt_init]), updates])), index=[times[0] - timedelta(7)] + list(times)[::7])

In [ ]:
n_samples = 10
proc_vals = idata.posterior["proc"].stack(sample=("chain", "draw")).values
init_vals = idata.posterior["rt_init"].stack(sample=("chain", "draw")).values
samples = np.random.choice(proc_vals.shape[1], size=n_samples, replace=False)
sampled_procs = proc_vals[:, samples].T
sampled_init = init_vals[samples]
result = np.concatenate([init_vals[samples][:, None], sampled_procs], axis=1)
pd.DataFrame(np.cumsum(result.T, axis=0), index=[times[0] - timedelta(7)] + list(times)[::7]).plot()

In [ ]:
get_process_from_updates(params["rt_init"], proc).plot()

In [ ]:
get_process_from_updates(sampled_init, sampled_vals).plot()

In [ ]:
# fig, axes = subplots(9, 7, figsize=[20, 16], sharex=True)
# flat_axes = axes.ravel()
# plot_width = 0.5
# for i in range(len(proc)):
#     ax = flat_axes[i]
#     fig = az.plot_posterior(idata.posterior["proc"][:, :, i], ax=ax, hdi_prob="hide", point_estimate=None)
#     fig.vlines(proc[i], 0.0, 1.5, color="k")
#     ax.set_title("")
#     ax.set_ylabel("")
#     ax.set_xlabel("")
#     ax.set_xlim([-plot_width, plot_width])
# flat_axes[-1].remove()
# fig = ax.get_figure()
# fig.tight_layout()

In [ ]:
fig = az.plot_trace(idata)
plt.tight_layout()